# 09 — Loss karşılaştırmaları

Aynı Git → GPU → Drive → hazırlık → eğitim → ZIP düzeni.
A–D'nin 38 fold × 3 seed sonuçları eksiksiz olmalı. Seçim ölçütü validation BCE;
kazanan A (6 özellik / GRU). Aynı 6 boyut, ham fiyat/ATR etiketleri ve splitler korunur.
Eylül 2026 test dönemi okunmaz; bütün ölçümler validation.

Önce 08 tamamlanmış olmalı. Wavelet kazananı otomatik sabitlenir. Saf BCE kontrolü seçilen A veya W_ON sonucundan alınır; yeniden eğitilmez. Ağırlıklı BCE, Focal, BCE + Pearson ve Focal + Pearson denenir.

GPU ve ortam cached kontrolle eşleşmelidir; uyumsuzlukta durur.
Tüm losslar için epoch ve yöntem seçimi ortak **validation BCE**, eşik **0,5**.
Parametreler önceden `configs/advisor_stages.yaml` içinde sabittir.
Checkpointler Drive'da; kesinti sonrası aynı commit/ayar/ortamla baştan çalıştırın.
Başka bir runtime'ı aynı çıktı alanına eşzamanlı yazdırmayın.

In [ ]:
import copy
import os, sys, subprocess, json, shutil, hashlib
from pathlib import Path
from datetime import datetime, timezone
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
os.environ['ADVISOR_LOCK_ROOT'] = '/content/advisor_runtime/locks'
import torch
assert torch.cuda.is_available(), 'Runtime → Change runtime type → GPU seçin.'
print('GPU:', torch.cuda.get_device_name(0), '| Torch:', torch.__version__)

# Bu notebookun varsayılan deneyleri. Sonuçlara bakarak seed/fold veya ölçüt değiştirmeyin.
EXPERIMENTS = ['L_WEIGHTED_BCE', 'L_FOCAL', 'L_BCE_PEARSON', 'L_FOCAL_PEARSON']
REPO_URL = 'https://github.com/umutergul74/yeniBot.git'
REPO_BRANCH = 'codex/advisor-ablation-protocol'
REPO_COMMIT = ''  # Boş: dalın güncel commit'i. Devam için eski commit'i buraya sabitleyebilirsiniz.
REPO_DIR = Path('/content/yenibot_advisor_repo')
DRIVE_BASE = Path('/content/drive/MyDrive/yeniBot')
LOCAL_BASE = Path('/content/advisor_runtime')
AUTO_UNASSIGN = False  # True: başarıyla eğitim + ZIP tamamlandıktan sonra GPU oturumunu bırakır.
PHASE = 'loss'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PERSIST = DRIVE_BASE / 'advisor_experiments' / 'colab_v1'
for folder in [PERSIST / 'inputs', PERSIST / 'data', PERSIST / 'runs', PERSIST / 'sessions', PERSIST / 'reports', LOCAL_BASE]:
    folder.mkdir(parents=True, exist_ok=True)
print('Kalıcı çıktı alanı:', PERSIST)

## Git'ten güncel deney kodunu çek

Doğru dal açıkça seçilir; commit ekrana ve sonuç paketine kaydedilir. Özel depo için Git kimlik
doğrulaması gerekirse mevcut hesabınızla erişim sağlayın; notebooka token yazmayın.
Çekilen kodu eğitim ayrı Python işleminde yükler. Önceki hücrelerden `yenibot` modülleri
yüklenmişse kodu güncellemeden önce oturumu yeniden başlatmanız istenir.

In [ ]:
def git(*args):
    return subprocess.check_output(['git', '-C', str(REPO_DIR), *args], text=True).strip()

if any(name == 'yenibot' or name.startswith('yenibot.') for name in sys.modules):
    raise RuntimeError('Kod hücresini tekrar çalıştırmadan önce Runtime → Restart session; sonra baştan çalıştırın.')
if not (REPO_DIR / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
else:
    assert not git('status', '--porcelain', '--untracked-files=no'), 'Colab çalışma kopyasında değişiklik var; otomatik silinmez.'
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_BRANCH], check=True)
    if REPO_COMMIT:
        subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--detach', REPO_COMMIT], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO_DIR), 'merge', '--ff-only', 'origin/' + REPO_BRANCH], check=True)
if REPO_COMMIT:
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--detach', REPO_COMMIT], check=True)
repo_commit = git('rev-parse', 'HEAD')
assert (REPO_DIR / 'yenibot/training/advisor.py').exists(), 'Yanlış dal veya eksik deney kodu.'
print('Dal:', REPO_BRANCH, '| Commit:', repo_commit)
sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)

In [ ]:
# Colab'ın mevcut GPU Torch paketini zorla değiştirmeyin; upgrade kullanılmıyor.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR / 'requirements.txt')], check=True)
import yaml
from yenibot.training.advisor import sha256, atomic_json
from yenibot.training.advisor_colab import (freeze_market_inputs, cache_frozen_inputs, create_review_bundle,
                                          freeze_oi_inputs, pin_completed_a_reference)
from yenibot.data import download_full_klines
from yenibot.data.binance import download_futures_metrics_from_vision
session_stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
SESSION = PERSIST / 'sessions' / session_stamp
SESSION.mkdir(parents=True, exist_ok=True)
packages = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
(SESSION / 'packages.txt').write_text(packages)
session_meta = {'session': session_stamp, 'branch': REPO_BRANCH, 'commit': repo_commit,
                'gpu': torch.cuda.get_device_name(0), 'python': sys.version, 'torch': torch.__version__,
                'experiments': EXPERIMENTS, 'status': 'setup', 'test_evaluations': 0}
atomic_json(SESSION / 'session.json', session_meta)
print('Ortam ve kaynak kaydı:', SESSION)
from yenibot.training.advisor_stages import select_architecture, select_wavelet
session_meta['phase'] = PHASE
atomic_json(SESSION / 'session.json', session_meta)

## Tamamlanmış kontrollere bağlan; sadece sabit geliştirme verisini hazırla

İndirme yapılmaz. Aynı Drive `colab_v1` alanındaki A–D ve market snapshot kullanılır.
Ham girdiler ile yeniden üretilen temel veri A ile hash düzeyinde eşleşmelidir.
Wavelet 256 geçmiş gözlem, db4, seviye 2, eşik ölçeği 0,5. Diğer dört girdi
ve etiketler değişmez. Oluşan iki kanallı ek veri bir kez Drive'a kaydedilir.
Açık pozisyon kaynağı bu altı özellikli modelde kullanılmadığından indirilmez.

In [ ]:
from yenibot.training.advisor_colab import cache_frozen_inputs, pin_completed_a_reference
stage_cfg = yaml.safe_load((REPO_DIR / 'configs/advisor_stages.yaml').read_text())
decision, control = select_architecture(PERSIST, stage_cfg)
print('Mimari seçimi:', decision['winner'], decision['architecture'], decision['features'])
print('A–D validation BCE:', {key: value['mean_bce'] for key,value in decision['references'].items()})
reference_a = pin_completed_a_reference(PERSIST)
snapshot = PERSIST / 'inputs/market_snapshot_v1'
assert (snapshot / 'snapshot_manifest.json').exists(), 'A deneyinin sabit snapshot alanı bulunamadı.'
cache_frozen_inputs(snapshot, LOCAL_BASE / 'raw')
cfg = copy.deepcopy(control['config'])
cfg['data']['snapshot'] = str(LOCAL_BASE / 'raw')
cfg['data']['feature_config'] = str(REPO_DIR / 'config.yaml')
cfg['data']['feature_columns_file'] = str(REPO_DIR / 'configs/advisor_full_features.txt')
cfg['output'] = str(PERSIST / 'runs')
frame_key = hashlib.sha256(json.dumps(stage_cfg['wavelet'],sort_keys=True).encode()).hexdigest()[:12]
source_key = sha256(REPO_DIR / 'yenibot/training/advisor_stages.py')[:12]
# Yeni kaynak imzası eski hazırlanmış frame'in üzerine yazmaz.
cfg['data']['frame'] = str(PERSIST / 'data' / f'stages_{frame_key}_{source_key}.parquet')
cfg['stage'] = {'phase': PHASE, 'specification': stage_cfg,
                'reference_manifest': str(reference_a), 'architecture_selection': decision,
                'holdout_config': str(REPO_DIR / 'configs/advisor_final_test.yaml')}
if PHASE == 'loss':
    choice = select_wavelet(PERSIST, stage_cfg, decision)
    cfg['stage']['wavelet_selection'] = choice
    print('Seçilmiş wavelet:', choice['winner'])
assert EXPERIMENTS and all(name == 'W_ON' if PHASE == 'wavelet' else name in stage_cfg['losses'] for name in EXPERIMENTS)
runtime_config = LOCAL_BASE / f'advisor_{PHASE}.yaml'
runtime_config.write_text(yaml.safe_dump(cfg,sort_keys=False))
shutil.copyfile(runtime_config, SESSION / runtime_config.name)
shutil.copyfile(REPO_DIR / 'configs/advisor_stages.yaml', SESSION / 'advisor_stages.yaml')
shutil.copyfile(REPO_DIR / 'configs/advisor_final_test.yaml', PERSIST / 'data/reserved_test_protocol.yaml')
print('Purge/embargo:', cfg['walk_forward']['purge_bars'],cfg['walk_forward']['embargo_bars'])
print('Checkpoint kapsamı:', EXPERIMENTS)

In [ ]:
def execute_logged(command, log_name):
    # Ayrı süreç güncel kodu yükler; hata kodu başarısızlığı gizlemez.
    with (SESSION / log_name).open('a', buffering=1) as log:
        with subprocess.Popen(command, cwd=REPO_DIR, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            try:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                return_code = process.wait()
            except BaseException:
                process.terminate()
                process.wait()
                raise
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)

In [ ]:
execute_logged([sys.executable, '-u', '-m', 'yenibot.training.advisor_stages', 'prepare',
                '--config', str(runtime_config)], 'prepare_stages.log')
manifest_path = Path(cfg['data']['frame']).with_suffix('.manifest.json')
prepared = json.loads(manifest_path.read_text())
print('Satır:', prepared['rows'], '| Tarih:', prepared['start'],prepared['end'])
print('Wavelet denetimi:', prepared['wavelet_audit'])
assert prepared['rows'] == 33551

## Eğitim ve kaldığı yerden devam

Her epoch'ta Drive checkpoint'i ve ilerleme kaydı yazılır. Bitmiş foldlar atlanır.
`runs/<deney>/STATUS.md` ve foldun `progress.json` dosyası kalan aşamayı gösterir.
09'da tek loss çalıştırmak için EXPERIMENTS listesini daraltabilirsiniz; sonrasında
varsayılan tüm listeyi çalıştırmak bitmişleri atlar ve eksikleri tamamlar.
Tüm losslar tamamlanmadan loss seçimi kaydedilmez. ZIP başarısızlıkta da hazırlanır.

In [ ]:
bundle_path = None
session_meta['status'] = 'training'
atomic_json(SESSION / 'session.json', session_meta)
try:
    for experiment in EXPERIMENTS:
        session_meta['current_experiment'] = experiment
        atomic_json(SESSION / 'session.json', session_meta)
        execute_logged([sys.executable, '-u', '-m', 'yenibot.training.advisor_stages', 'run',
                        '--config', str(runtime_config), '--experiment', experiment], f'train_{experiment}.log')
    if PHASE == 'loss' and set(EXPERIMENTS) == set(stage_cfg['losses']):
        execute_logged([sys.executable, '-u', '-m', 'yenibot.training.advisor_stages', 'finalize',
                        '--config', str(runtime_config)], 'loss_selection.log')
    elif PHASE == 'loss':
        print('Seçili alt kapsam bitti; tüm losslar tamamlanmadan kazanan seçilmez.')
    session_meta['status'] = 'complete' 
    session_meta.pop('current_experiment', None)
except BaseException as exc:
    session_meta['status'] = 'interrupted_or_failed'
    session_meta['error'] = repr(exc)
    raise
finally:
    session_meta['updated_at'] = datetime.now(timezone.utc).isoformat()
    atomic_json(SESSION / 'session.json', session_meta)
    bundle_path = create_review_bundle(PERSIST, session_meta)
    print('Drive inceleme ZIP:', bundle_path)
    print('İşlem durumu:', session_meta['status'])

In [ ]:
# Her klasörün kapsamını ayrı göster: tamamlanmamış sonuçlardan kazanan seçme.
for directory in sorted((PERSIST / 'runs').glob('*')):
    summary = directory / 'validation_summary.csv'
    status_path = directory / 'status.json'
    if not status_path.exists():
        continue
    status = json.loads(status_path.read_text())
    print(directory.name, '| durum:', status['status'], '| tamamlanan:', len(status['completed']),
          '| test değerlendirmesi:', status['test_evaluations'])
    if summary.exists():
        display(pd.read_csv(summary))
print('İnceleme için paylaşılacak küçük ZIP:', PERSIST / 'reports/advisor_latest_review_bundle.zip')
if AUTO_UNASSIGN and session_meta['status'] == 'complete' and bundle_path and bundle_path.exists():
    from google.colab import runtime
    runtime.unassign()

## Sonuçları burada incelememiz için

Drive: `MyDrive/yeniBot/advisor_experiments/colab_v1/reports/advisor_latest_review_bundle.zip`.
ZIP'i buraya ekleyebilirsiniz veya bilgisayara senkronize edilen dosyanın yolunu paylaşabilirsiniz.
Bu sohbetin Colab'a bağlanan Drive'ı kendiliğinden görmesi beklenmemelidir.
ZIP; commit, ayarlar, ortam sürümleri, sınır takvimi, epoch geçmişi, validation
tahminleri ve ilerleme durumunu içerir. Büyük model ağırlıkları Drive'da kalır.
Yerel deney ile Colab deneyi farklı imzalarla korunur; sonuçlar tek deneymiş gibi birleştirilmez.